In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Thesis Claim Analyzer - Example Notebook\n",
    "\n",
    "This notebook demonstrates how to use the Thesis Claim Analyzer to extract and analyze claims from research papers."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Install dependencies (run once)\n",
    "!pip install torch transformers datasets pymupdf spacy pandas numpy scikit-learn scipy requests pyyaml tqdm openpyxl\n",
    "!python -m spacy download en_core_web_sm"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Import libraries\n",
    "import sys\n",
    "from pathlib import Path\n",
    "import pandas as pd\n",
    "\n",
    "# Add project to path (adjust path as needed)\n",
    "sys.path.insert(0, '/content/thesis-claim-analyzer')\n",
    "\n",
    "from src.pipeline.analyzer import ThesisClaimAnalyzer"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Upload a PDF file\n",
    "from google.colab import files\n",
    "uploaded = files.upload()\n",
    "\n",
    "# Get the filename\n",
    "pdf_file = list(uploaded.keys())[0]\n",
    "print(f\"Uploaded: {pdf_file}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Initialize analyzer\n",
    "# use_ml=False for keyword-only (no training needed)\n",
    "# use_ml=True for ML-enhanced (requires trained model)\n",
    "analyzer = ThesisClaimAnalyzer(use_ml=False)\n",
    "print(\"✅ Analyzer initialized\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Run analysis\n",
    "analysis = analyzer.analyze_paper(pdf_file)\n",
    "print(f\"\\n📊 Found {len(analysis.claims)} claims\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Display sample claims\n",
    "print(\"\\n\" + \"=\"*80)\n",
    "print(\"SAMPLE CLAIMS\")\n",
    "print(\"=\"*80)\n",
    "\n",
    "for i, claim in enumerate(analysis.claims[:10], 1):\n",
    "    print(f\"\\n{i}. [{claim.claim_type}] Trust: {claim.trust_score:.0%}\")\n",
    "    print(f\"   {claim.text[:150]}...\")\n",
    "    print(f\"   Section: {claim.section} | Importance: {claim.importance}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Export to Excel\n",
    "output_file = \"analysis_results.xlsx\"\n",
    "analyzer.export_to_excel(analysis, output_file)\n",
    "print(f\"✅ Results saved to {output_file}\")\n",
    "\n",
    "# Download the file\n",
    "files.download(output_file)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Create a DataFrame for analysis\n",
    "claims_df = pd.DataFrame([c.to_dict() for c in analysis.claims])\n",
    "\n",
    "print(\"\\n\" + \"=\"*80)\n",
    "print(\"CLAIM STATISTICS\")\n",
    "print(\"=\"*80)\n",
    "\n",
    "print(f\"\\nTotal claims: {len(claims_df)}\")\n",
    "print(f\"\\nClaim types:\")\n",
    "print(claims_df['claim_type'].value_counts())\n",
    "print(f\"\\nImportance levels:\")\n",
    "print(claims_df['importance'].value_counts())\n",
    "print(f\"\\nAverage trust score: {claims_df['trust_score'].mean():.3f}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Visualize claim types\n",
    "import matplotlib.pyplot as plt\n",
    "\n",
    "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "\n",
    "# Claim types\n",
    "type_counts = claims_df['claim_type'].value_counts()\n",
    "axes[0].bar(type_counts.index, type_counts.values)\n",
    "axes[0].set_title('Claim Types')\n",
    "axes[0].tick_params(axis='x', rotation=45)\n",
    "\n",
    "# Trust score distribution\n",
    "axes[1].hist(claims_df['trust_score'], bins=20, edgecolor='black')\n",
    "axes[1].set_title('Trust Score Distribution')\n",
    "axes[1].set_xlabel('Trust Score')\n",
    "axes[1].set_ylabel('Count')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Filter high-value claims\n",
    "high_trust = claims_df[claims_df['trust_score'] > 0.7]\n",
    "critical = claims_df[claims_df['importance'] == 'Critical']\n",
    "\n",
    "print(f\"\\nHigh trust claims (>70%): {len(high_trust)}\")\n",
    "print(f\"Critical claims: {len(critical)}\")\n",
    "\n",
    "if len(high_trust) > 0:\n",
    "    print(\"\\nTop high-trust claims:\")\n",
    "    for _, row in high_trust.head(5).iterrows():\n",
    "        print(f\"  - {row['claim_type']}: {row['text'][:100]}...\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Save claims to CSV for further analysis\n",
    "claims_df.to_csv(\"all_claims.csv\", index=False)\n",
    "files.download(\"all_claims.csv\")\n",
    "print(\"✅ Claims exported to CSV\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.9.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}